# GloVe (mean pooling, concatenated) + bigger MLP

**Task:** Abductive Commonsense Reasoning (ART). Given two observations O1, O2 and two
hypotheses H1, H2, pick the hypothesis that better explains what happened in between.

**Approach of this notebook:** Same framework as `w2v_mean_mlp.ipynb`, three changes:

- glove vocabulary (`glove-wiki-gigaword-300`, uncased) -> tokens are lowercased, much lower OOV
- the four sentence vectors are concatenated in fixed order O1, O2, H1, H2 instead of averaged -> 1200 dims, swapping H1 and H2 changes the input
- one more linear layer + ReLU in the MLP

In [1]:
%pip install -q datasets gensim lightning nltk python-dotenv wandb

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from collections import Counter
from pathlib import Path

import gensim.downloader as api
import lightning as L
import nltk
import numpy as np
import torch
import torch.nn as nn
import wandb
from datasets import load_dataset
from dotenv import find_dotenv, load_dotenv
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger
from torch.utils.data import DataLoader, TensorDataset

nltk.download("punkt_tab", quiet=True)

True

In [3]:
dotenv_path = find_dotenv(usecwd=True)
load_dotenv(dotenv_path)
print(f"environment loaded from: {dotenv_path or '<no .env found, using the defaults below>'}")

WANDB_BASE_URL = os.getenv("WANDB_BASE_URL", "https://api.wandb.ai")
WANDB_ENTITY = os.getenv("WANDB_ENTITY", "cmedi32-hochschule-luzern")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "nlp-project-01-abductive-reasoning")

os.environ["WANDB_BASE_URL"] = WANDB_BASE_URL
os.environ["WANDB_ENTITY"] = WANDB_ENTITY
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

wandb.login()

environment loaded from: /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/.env


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: cmedi32 (cmedi32-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
config = {
    "dataset": "amarfurt/art-nlp-hs26",
    "tokenizer": "nltk.word_tokenize",
    "lowercase": True,
    "remove_punctuation": True,
    "remove_stopwords": False,
    "normalization": "none",
    "truncation": "none",
    "embedding": "glove-wiki-gigaword-300",
    "sentence_representation": "mean",
    "example_representation": "concat",  # O1, O2, H1, H2
    "classifier": "mlp_2_hidden",
    "hidden_dim": 256,
    "learning_rate": 1e-3,
    "batch_size": 64,
    "max_epochs": 100,
    "patience": 5,  # validation checks without improvement of loss/valid before stopping
    "seed": 42,
}

run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name="glove-concatenate-bigger-mlp",
    config=config,
)

In [5]:
dataset = load_dataset("amarfurt/art-nlp-hs26")

train = dataset["train"]
valid = dataset["validation"]
test = dataset["test"]

print(f"train: {len(train)}, valid: {len(valid)}, test: {len(test)}")

train: 16801, valid: 1000, test: 1532


In [6]:
def preprocess(text):
    tokens = nltk.word_tokenize(text)
    return [token.lower() for token in tokens if any(character.isalnum() for character in token)]

In [7]:
TEXT_FIELDS = ["observation_1", "observation_2", "hypothesis_1", "hypothesis_2"]

def preprocess_example(example):
    processed = {field: preprocess(example[field]) for field in TEXT_FIELDS}
    processed["target"] = example["label"] - 1
    return processed


def preprocess_split(split):
    return split.map(preprocess_example)

In [8]:
train_tokens = preprocess_split(train)
valid_tokens = preprocess_split(valid)
test_tokens = preprocess_split(test)

Map:   0%|          | 0/16801 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1532 [00:00<?, ? examples/s]

In [9]:
wv = api.load(config["embedding"])
print(f"vocabulary: {len(wv.index_to_key)} words, {wv.vector_size} dimensions")

vocabulary: 400000 words, 300 dimensions


In [10]:
oov_counts = Counter()
embed_stats = Counter()

def embed(tokens):
    vectors = []
    for token in tokens:
        embed_stats["tokens"] += 1
        if token in wv:
            vectors.append(wv[token])
        else:
            embed_stats["oov"] += 1
            oov_counts[token] += 1
    if not vectors:
        embed_stats["empty_sentences"] += 1
        return np.zeros(wv.vector_size)
    return np.mean(vectors, axis=0)

In [11]:
def embed_example(example):
    vectors = [embed(example[field]) for field in TEXT_FIELDS]
    return np.concatenate(vectors) # @todo apply tfidf with embeddings, and combine the information

In [12]:
X_train = np.stack([embed_example(example) for example in train_tokens])
y_train = np.array(train_tokens["target"])
X_valid = np.stack([embed_example(example) for example in valid_tokens])
y_valid = np.array(valid_tokens["target"])
X_test = np.stack([embed_example(example) for example in test_tokens])
y_test = np.array(test_tokens["target"])

### DataLoaders - Takes care of Training Batching and Loading

In [13]:
L.seed_everything(config["seed"])

# @todo this has to be optimized or at least it has to be documented how its optimized with badge sizing

def make_loader(X, y, shuffle):
    dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    return DataLoader(dataset, batch_size=config["batch_size"], shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
valid_loader = make_loader(X_valid, y_valid, shuffle=False)
test_loader = make_loader(X_test, y_test, shuffle=False)

Seed set to 42


### LightningModule Model (Derived form libraries.pdf using coding agents in combination with Lightning Documentation)

In [14]:
class MLP(L.LightningModule):

    def __init__(self, input_dim, hidden_dim, output_dim, learning_rate):
        super().__init__()
        self.save_hyperparameters()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def reset_parameters(self):
        for layer in self.layers:
            if isinstance(layer, nn.Linear):
                layer.reset_parameters()

    def forward(self, x):
        return self.layers(x)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate) # investigate and evaluate learning rate

    def shared_step(self, batch, split):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        accuracy = (logits.argmax(dim=1) == y).float().mean()
        self.log(f"loss/{split}", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"accuracy/{split}", accuracy, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    # check that against ML libraries
    def training_step(self, batch, batch_idx):
        return self.shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self.shared_step(batch, "valid")

    def test_step(self, batch, batch_idx):
        return self.shared_step(batch, "test")

In [15]:
WEIGHTS_DIR = Path("../../../models/project_1/glove_concatenate_bigger_mlp/weights")

model = MLP(
    input_dim=len(TEXT_FIELDS) * wv.vector_size,
    hidden_dim=config["hidden_dim"],
    output_dim=2,
    learning_rate=config["learning_rate"],
)
model.reset_parameters()

early_stopping = EarlyStopping(monitor="loss/valid", mode="min", patience=config["patience"], verbose=True) # check against documentation and understand it
checkpoint = ModelCheckpoint(dirpath=WEIGHTS_DIR, filename="best-{epoch}", monitor="loss/valid", mode="min", save_top_k=1) # same here

trainer = L.Trainer( # train
    max_epochs=config["max_epochs"],
    accelerator="auto",
    devices=1,
    logger=WandbLogger(),  # reuses the run from wandb.init
    callbacks=[early_stopping, checkpoint], # ensure it stops when performance degrades
)
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/loggers/wandb.py:400: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ layers  │ Sequential       │  373 K │ train │     0 │
│ 1 │ loss_fn │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 373 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 373 K                                                                                                
Total estimated model params size (MB): 1.495                                                                      
Modules in train mode: 7                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, 
LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.

Metric loss/valid improved. New best score: 0.672
Metric loss/valid improved by 0.007 >= min_delta = 0.0. New best score: 0.665
Metric loss/valid improved by 0.004 >= min_delta = 0.0. New best score: 0.661
Metric loss/valid improved by 0.008 >= min_delta = 0.0. New best score: 0.653
Monitored metric loss/valid did not improve in the last 5 records. Best score: 0.653. Signaling Trainer to stop.


### Best Checkpoint + Upload Artifacts to Weights and Biases

In [16]:
trainer.validate(model, dataloaders=valid_loader, ckpt_path="best")
trainer.test(model, dataloaders=test_loader, ckpt_path="best")

run.summary.update({
    "accuracy/majority": np.bincount(y_test).max() / len(y_test),
    "best/loss_valid": checkpoint.best_model_score.item(),
    "best/checkpoint": Path(checkpoint.best_model_path).name,
    "early_stopping/stopped_epoch": early_stopping.stopped_epoch,
    "oov/tokens": embed_stats["tokens"],
    "oov/occurrences": embed_stats["oov"],
    "oov/rate": embed_stats["oov"] / embed_stats["tokens"],
    "oov/unique_types": len(oov_counts),
    "oov/empty_sentences": embed_stats["empty_sentences"],
})

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_bigger_mlp/weights/best-epoch=3.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_bigger_mlp/weights/best-epoch=3.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      accuracy/valid       │    0.5979999899864197     │
│        loss/valid         │    0.6527876257896423     │
└───────────────────────────┴───────────────────────────┘

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_bigger_mlp/weights/best-epoch=3.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_bigger_mlp/weights/best-epoch=3.ckpt


Output()

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       accuracy/test       │    0.5169712901115417     │
│         loss/test         │    0.7795184850692749     │
└───────────────────────────┴───────────────────────────┘

In [17]:
artifact = wandb.Artifact("glove-concatenate-bigger-mlp", type="model", metadata={"loss/valid": checkpoint.best_model_score.item()})
artifact.add_file(checkpoint.best_model_path)
run.log_artifact(artifact, aliases=["best"])
wandb.finish()

accuracy/test,▁
accuracy/train,▁▃▄▄▅▅▆▇█
accuracy/valid,▁▆▇▇▇█▆▆▁▇
epoch,▁▁▂▂▃▃▃▃▄▄▅▅▆▆▆▆▇▇██
loss/test,▁
loss/train,█▇▆▆▅▅▄▂▁
loss/valid,▂▂▁▁▂▃▃▅█▁
trainer/global_step,▁▁▂▂▃▃▄▄▄▄▅▅▆▆▇▇████
accuracy/majority,0.50979
accuracy/test,0.51697
accuracy/train,0.76359
